In [1]:
import pandas as pd
import numpy as np
import os
import re
from antropy import sample_entropy
from tqdm import tqdm

In [2]:
# combining 
dates = ['2025-03-03', '2025-03-04', '2025-03-05', '2025-03-06', '2025-03-07']

all_data = []

for date in dates:

    base_path = f'/Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/{date}'

    for file in os.listdir(base_path):
        if file.endswith(".csv"):

            file_path = os.path.join(base_path, file)
            print(f"Processing: {date} -> {file}")

            df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')

            keep_cols = [
                'class',
                'Time_In_PST',
                'Rotation X',
                'Rotation Y',
                'Rotation Z',
                'Rotation W',
                'Acceleration X',
                'Acceleration Y',
                'Acceleration Z'
            ]

            df = df[[col for col in df.columns if col in keep_cols]]

            for col in keep_cols:
                if col not in df.columns:
                    df[col] = pd.NA

            df = df[keep_cols]

            match = re.search(r'(AnkleL|AnkleR|WristL|WristR|Head|Hip)', file)
            df['Sensor'] = match.group(0) if match else file
            df['Date'] = date

            all_data.append(df)

combined = pd.concat(all_data, ignore_index=True)

output_path = '/Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/P009_dates_combined.csv'
combined.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Processing: 2025-03-03 -> P009MocopiWristLDeviceOne2025-03-03.csv
Processing: 2025-03-03 -> P009MocopiAnkleRDeviceOne2025-03-03.csv
Processing: 2025-03-03 -> P009MocopiHipDeviceOne2025-03-03.csv
Processing: 2025-03-03 -> P009MocopiHeadDeviceOne2025-03-03.csv
Processing: 2025-03-03 -> P009MocopiAnkleLDeviceOne2025-03-03.csv
Processing: 2025-03-03 -> P009MocopiWristRDeviceOne2025-03-03.csv
Processing: 2025-03-04 -> P009MocopiAnkleRDeviceOne2025-03-04.csv
Processing: 2025-03-04 -> P009MocopiWristLDeviceOne2025-03-04.csv
Processing: 2025-03-04 -> P009MocopiHipDeviceOne2025-03-04.csv
Processing: 2025-03-04 -> P009MocopiHeadDeviceOne2025-03-04.csv
Processing: 2025-03-04 -> P009MocopiWristRDeviceOne2025-03-04.csv
Processing: 2025-03-04 -> P009MocopiAnkleLDeviceOne2025-03-04.csv
Processing: 2025-03-05 -> P009MocopiWristLDeviceOne2025-03-05.csv
Processing: 2025-03-05 -> P009MocopiAnkleRDeviceOne2025-03-05.csv
Processing: 2025-03-05 -> P009MocopiHipDeviceOne2025-03-05.csv
Processing: 2025-03-05 

In [3]:
# creating accel/jerk magnitudes
import numpy as np
import pandas as pd

input_file = '/Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/P009_dates_combined.csv'

combined = pd.read_csv(input_file)

combined = combined.sort_values(['Sensor', 'Date', 'Time_In_PST']).reset_index(drop=True)

# Acceleration Magnitude
combined['Acc_Mag'] = np.sqrt(
    combined['Acceleration X']**2 +
    combined['Acceleration Y']**2 +
    combined['Acceleration Z']**2
)

# Jerk (difference of acceleration)
combined['Jerk_X'] = combined.groupby(['Sensor', 'Date'])['Acceleration X'].diff()
combined['Jerk_Y'] = combined.groupby(['Sensor', 'Date'])['Acceleration Y'].diff()
combined['Jerk_Z'] = combined.groupby(['Sensor', 'Date'])['Acceleration Z'].diff()

combined['Jerk_Mag'] = np.sqrt(
    combined['Jerk_X']**2 +
    combined['Jerk_Y']**2 +
    combined['Jerk_Z']**2
)

# remove uneeded columns
combined = combined.drop(columns=[
    'Acceleration X',
    'Acceleration Y',
    'Acceleration Z',
    'Rotation X',
    'Rotation Y',
    'Rotation Z',
    'Rotation W',
    'Jerk_X',
    'Jerk_Y',
    'Jerk_Z'
])

output_path = '/Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/P009_features.csv'
combined.to_csv(output_path, index=False)

print(f"New dataset saved to: {output_path}")

New dataset saved to: /Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/P009_features.csv


In [4]:
df = pd.read_csv("P009_features.csv")
df["Participant"] = "P009"

# sort once (important for time series integrity)
df = df.sort_values(['class', 'Sensor', 'Date', 'Time_In_PST'])

df = df.iloc[::3].copy()

def compute_sampen(series):
    series = np.asarray(series)          # ensure numpy array
    series = series[~np.isnan(series)]   # remove NaNs safely

    # skip too-short signals
    if len(series) < 50:
        return np.nan

    # speed cap
    series = series[:300]

    try:
        return sample_entropy(series, order=2, metric='chebyshev')
    except:
        return np.nan


results = []

group_cols = ['class', 'Sensor', 'Date']
grouped = df.groupby(group_cols)

for keys, group in tqdm(grouped, total=grouped.ngroups):

    acc_sampen = compute_sampen(group['Acc_Mag'].values)
    jerk_sampen = compute_sampen(group['Jerk_Mag'].values)

    results.append({
        'participant': "P009",
        'class': keys[0],
        'Sensor': keys[1],
        'Date': keys[2],
        'Acc_SampEn': acc_sampen,
        'Jerk_SampEn': jerk_sampen,
        'n_points': len(group)   
    })

sampen_df = pd.DataFrame(results)

# save final dataset
sampen_df.to_csv(
    "/Users/jaimesong/Desktop/Research/hypersense/All/GreenClass/P009/P009_sampen_features.csv",
    index=False
)
print(sampen_df.head())

100%|██████████| 145/145 [00:01<00:00, 104.11it/s]

  participant class  Sensor        Date  Acc_SampEn  Jerk_SampEn  n_points
0        P009   ELA  AnkleL  2025-03-03    0.095875     0.094596     77645
1        P009   ELA  AnkleL  2025-03-04    0.699782     0.915008     93283
2        P009   ELA  AnkleL  2025-03-05    0.197561     0.137569     24063
3        P009   ELA  AnkleL  2025-03-06    1.144522     1.187684     92093
4        P009   ELA  AnkleL  2025-03-07    0.129212     0.118450      4141
